In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLO11 segmentation model
model = YOLO("yolo11s.pt")

# Train with enhanced parameters for plant diseases
results = model.train(
    data="./dataset/data.yaml",
    epochs=100,
    imgsz=736,
    task="detect",
    batch=4,
    device=0,
    patience=15,

    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.2,
    shear=2.0,
    flipud=0.5,
    fliplr=0.5,
    copy_paste = 0.4,
    mosaic=0.8,
    mixup=0.1,

    name="plant_disease_detect"
)


In [ ]:
from ultralytics import YOLO, SAM
import os
import cv2
import numpy as np


yolo_model = YOLO("./runs/detect/plant_disease_detect/weights/best.pt")
sam_model = SAM("sam2_b.pt")

In [ ]:
import os
input_folder = "./dataset/train/images"
output_images = "./content/auto_dataset/train/images"
output_labels = "./content/auto_dataset/train/labels"

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_labels, exist_ok=True)

In [ ]:
def mask_to_polygon(mask):
    contours, _ = cv2.findContours(
        mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    if len(contours) == 0:
        return None

    contour = max(contours, key=cv2.contourArea)
    contour = contour.squeeze()

    if len(contour.shape) != 2:
        return None

    return contour

In [ ]:
CONF_THRESHOLD = 0.4
MIN_MASK_AREA = 50
MIN_POLYGON_POINTS = 3
OVERLAP = 0.2

# ------------------ PIPELINE ------------------
image_files = os.listdir(input_folder)

for img_name in image_files:
    try:
        img_path = os.path.join(input_folder, img_name)

        # YOLO inference
        result = yolo_model(img_path)[0]

        if result.boxes is None:
            continue

        image = result.orig_img
        h, w = image.shape[:2]

        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        # -------- FILTER DETECTIONS --------
        filtered_boxes = []
        filtered_classes = []

        for box, score, cls in zip(boxes, scores, classes):
            if score >= CONF_THRESHOLD:
                filtered_boxes.append(box)
                filtered_classes.append(int(cls))

        if len(filtered_boxes) == 0:
            continue

        # -------- SAM SEGMENTATION --------
        sam_result = sam_model(
            source=img_path,
            bboxes=filtered_boxes,
            save=False
        )[0]

        if sam_result.masks is None:
            continue

        masks = sam_result.masks.data.cpu().numpy()

        # -------- Separate leaf masks (IMPORTANT) --------
        # Assuming class 0 = leaf
        leaf_masks = [m for m, c in zip(masks, filtered_classes) if c == 0]

        # -------- SAVE IMAGE --------
        cv2.imwrite(os.path.join(output_images, img_name), image)

        label_lines = []

        # -------- PROCESS EACH MASK --------
        for mask, cls in zip(masks, filtered_classes):

            # 0. Safety check
            if mask.sum() == 0:
                continue

            # 1. Remove small noisy masks
            if mask.sum() < MIN_MASK_AREA:
                continue

            # -------- LEAF-DISEASE CONSTRAINT --------
            # Assuming class 1 = disease
            if cls >= 1:
                valid = False

                for leaf_mask in leaf_masks:
                    overlap = np.logical_and(mask, leaf_mask).sum()

                    if overlap / (mask.sum() + 1e-6) > OVERLAP:
                        valid = True
                        break

                if not valid:
                    continue
            # ----------------------------------------

            # 2. Convert to polygon
            polygon = mask_to_polygon(mask)
            if polygon is None:
                continue

            # 3. Remove weak polygons
            if len(polygon) < MIN_POLYGON_POINTS:
                continue

            # 4. Normalize coordinates
            poly_norm = []
            for x, y in polygon:
                poly_norm.append(x / w)
                poly_norm.append(y / h)

            # 5. Final label line
            line = f"{cls} " + " ".join(map(str, poly_norm))
            label_lines.append(line)

        # -------- SAVE LABEL FILE --------
        if len(label_lines) > 0:
            label_path = os.path.join(
                output_labels,
                os.path.splitext(img_name)[0] + ".txt"
            )
            with open(label_path, "w") as f:
                f.write("\n".join(label_lines))

    except Exception as e:
        print(f"Error processing {img_name}: {e}")

In [ ]:
import os

def clean_dataset(image_folder, label_folder):
    # Get lists of all files
    image_files = {os.path.splitext(f)[0]: f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
    label_files = {os.path.splitext(f)[0] for f in os.listdir(label_folder) if f.endswith('.txt')}

    count = 0
    
    # Iterate through images and check for labels
    for img_id, img_filename in image_files.items():
        if img_id not in label_files:
            img_path = os.path.join(image_folder, img_filename)
            try:
                os.remove(img_path)
                print(f"Removed image with missing label: {img_filename}")
                count += 1
            except Exception as e:
                print(f"Error deleting {img_filename}: {e}")

    print(f"\nCleanup complete. Removed {count} images.")

# --- RUN THIS ---
clean_dataset(output_images, output_labels)


In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLO11 segmentation model
model = YOLO("yolo11s-seg.pt")

# Train with enhanced parameters for plant diseases
results = model.train(
    data="./content/auto_dataset/data.yaml",
    epochs=150,
    imgsz=736,
    batch=4,
    device=0,
    patience=15,

    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.2,
    shear=2.0,
    flipud=0.5,
    fliplr=0.5,
    copy_paste = 0.4,
    mosaic=0.8,
    mixup=0.1,
    overlap_mask = False,
    close_mosaic=30,
    overlap_mask = False,
    close_mosaic=50,

    cls=1.5,       
    box=7.5,

    name="plant_disease_segment"
)

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# 1. Setup paths
source_paths = ['./test/test1.jpg', './test/test2.jpg', './test/test3.jpg', './test/test4.jpg']
model = YOLO("./runs/segment/plant_disease_segment4/weights/best.pt")

# --- SAVE ORIGINAL 2x2 GRID ---
# Load original images from disk
original_imgs = [cv2.imread(p) for p in source_paths]

# Resize to ensure they match (using first image dimensions)
h, w, _ = original_imgs[0].shape
resized_originals = [cv2.resize(img, (w, h)) for img in original_imgs]

# Stack into 2x2 grid
orig_top = np.hstack((resized_originals[0], resized_originals[1]))
orig_bottom = np.hstack((resized_originals[2], resized_originals[3]))
original_grid = np.vstack((orig_top, orig_bottom))

# Save original collage
cv2.imwrite('original_batch_grid.jpg', original_grid)
print("Original grid saved as 'original_batch_grid.jpg'")

# --- RUN INFERENCE & SAVE ANNOTATED GRID ---
results = model.predict(source=source_paths, conf=0.5, iou=0.3, save=False, agnostic_nms=True)
annotated_imgs = [r.plot(masks=True, line_width=2) for r in results]

# Stack annotated images
ann_top = np.hstack((annotated_imgs[0], annotated_imgs[1]))
ann_bottom = np.hstack((annotated_imgs[2], annotated_imgs[3]))
annotated_grid = np.vstack((ann_top, ann_bottom))

cv2.imwrite('annotated_batch_grid.jpg', annotated_grid)
print("Annotated grid saved as 'annotated_batch_grid.jpg'")


In [ ]:
from ultralytics import YOLO, SAM
import cv2
import numpy as np

# -------------------- CONFIG --------------------
CONF_THRESHOLD = 0.2
LEAF_MASK_T = 0.2
DISEASE_MASK_T = 0.2

MIN_SAM_AREA = 25
OVERLAP = 0.5  # disease-mask inside leaf constraint

# -------------------- MODELS --------------------
yolo_model = YOLO("./runs/segment/plant_disease_segment4/weights/best.pt")
sam_model = SAM("sam2_b.pt")

# -------------------- READ IMAGE --------------------
img_path = "./original_batch_grid.jpg"
image = cv2.imread(img_path)
if image is None:
    raise FileNotFoundError(f"Could not read image: {img_path}")

# -------------------- YOLO INFERENCE --------------------
results = yolo_model.predict(source=img_path, conf=0.5, iou=0.3, save=False, agnostic_nms=True, imgsz=1024)
result = results[0]

annotated = result.plot(masks=True, line_width=2)

if result.masks is None or result.boxes is None:
    cv2.imshow("YOLO | Leaf | SAM (Disease)", cv2.resize(annotated, (1000, 350)))
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    raise SystemExit("No YOLO masks/boxes found.")

masks = result.masks.data.cpu().numpy()                  # (N, Hmask, Wmask) (YOLO mask logits/probs)
classes = result.boxes.cls.cpu().numpy().astype(int)    # (N,)
boxes_xyxy = result.boxes.xyxy.cpu().numpy()             # (N,4) in original image coords

h, w = image.shape[:2]

leaf_count = 0

# -------------------- MAIN LOOP: per LEAF --------------------
for i, cls_id in enumerate(classes):
    if int(cls_id) != 0:  # leaf only
        continue

    leaf_count += 1

    # Leaf mask -> binary -> resize to image size (safe)
    leaf_mask = (masks[i] > LEAF_MASK_T).astype(np.uint8)
    leaf_mask = cv2.resize(leaf_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    if leaf_mask.sum() < 50:
        continue

    # Apply leaf mask to get leaf-only image (optional)
    leaf_only = image.copy()
    leaf_only[leaf_mask == 0] = 0

    # Tight bbox from leaf mask
    y_idx, x_idx = np.where(leaf_mask == 1)
    if len(y_idx) == 0 or len(x_idx) == 0:
        continue

    y1, y2 = y_idx.min(), y_idx.max()
    x1, x2 = x_idx.min(), x_idx.max()

    # Crop (leaf crop is what SAM will receive)
    crop = image[y1:y2 + 1, x1:x2 + 1]
    crop_leaf_mask = leaf_mask[y1:y2 + 1, x1:x2 + 1]
    

    # -------------------- Collect YOLO DISEASE boxes inside this leaf --------------------
    sam_bboxes_crop = []

    for j, cls_j in enumerate(classes):
        if int(cls_j) != 1:  # disease only
            continue

        disease_mask = (masks[j] > DISEASE_MASK_T).astype(np.uint8)
        disease_mask = cv2.resize(disease_mask, (w, h), interpolation=cv2.INTER_NEAREST)

        if disease_mask.sum() < 50:
            continue

        # Overlap constraint: disease mask must significantly overlap the leaf mask
        overlap = np.logical_and(disease_mask == 1, leaf_mask == 1).sum()
        if overlap / (disease_mask.sum() + 1e-6) <= OVERLAP:
            continue

        # YOLO disease bbox in full-image coords
        dx1, dy1, dx2, dy2 = boxes_xyxy[j]
        dx1, dy1, dx2, dy2 = map(int, [dx1, dy1, dx2, dy2])

        # Convert full-image bbox -> crop coords and clip
        bx1 = max(dx1, x1) - x1
        by1 = max(dy1, y1) - y1
        bx2 = min(dx2, x2) - x1
        by2 = min(dy2, y2) - y1

        if bx2 <= bx1 or by2 <= by1:
            continue

        sam_bboxes_crop.append([bx1, by1, bx2, by2])

    if len(sam_bboxes_crop) == 0:
        continue

    # -------------------- SAM on cropped leaf guided by disease boxes --------------------
    sam_result = sam_model(
        source=crop,
        bboxes=sam_bboxes_crop,
        save=False
    )[0]

    sam_overlay = crop.copy()

    if sam_result.masks is not None:
        sam_masks = sam_result.masks.data.cpu().numpy()  # (M, cropH, cropW)

        for m in sam_masks:
            m_bin = (m > 0.5).astype(np.uint8)

            # noise filtering
            if m_bin.sum() < MIN_SAM_AREA:
                continue

            # optional: ensure SAM mask lies on/inside the leaf crop
            overlap2 = np.logical_and(m_bin == 1, crop_leaf_mask == 1).sum()
            if overlap2 / (m_bin.sum() + 1e-6) <= OVERLAP:
                continue

            # Overlay in RED (disease region)
            sam_overlay[m_bin == 1] = [0, 0, 255]

    # -------------------- Visualization (same style as your code) --------------------
    left = cv2.resize(annotated, (350, 350))
    middle = cv2.resize(crop, (350, 350))
    right = cv2.resize(sam_overlay, (350, 350))

    cv2.putText(left, "YOLO", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.putText(middle, f"Leaf {leaf_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.putText(right, "SAM Disease", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    combined = np.hstack((left, middle, right))
    combined_small = cv2.resize(combined, (1000, 350))

    cv2.imshow("YOLO | Leaf (crop) | SAM (disease)", combined_small)
    key = cv2.waitKey(0)
    if key == ord('q'):
        break

cv2.destroyAllWindows()
